# Gemma 4 E2B 1000 campaign: Aggregator (CPU)

Combines the two GPU workers' live state (monitor/gemma/workers/w1.*, w2.*) into the canonical monitor/gemma/state.json + history.jsonl + live.json the gemma dashboard page reads. Runs every 45s for the whole campaign. CPU-only -- does not count against Kaggle's 2-concurrent-GPU limit. If this session dies, relaunch it; nothing is lost.

## 1. Secrets (hardcoded env vars)

In [ ]:
import os
print("secrets are injected at build time; this cell is a placeholder")

## 2. Get the repo

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "chess-slm-benchmark"
if REPO.exists():
    shutil.rmtree(REPO)

def find_token():
    for name in ("GITHUB_TOKEN", "GH_TOKEN"):
        if os.environ.get(name):
            return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None

token = find_token()
url = "https://github.com/Vedang-P/chess-slm-benchmark.git"
if token:
    url = url.replace("https://", f"https://x-access-token:{token}@")
# --local-thinking, --live-namespace, scored-state live publishing and the
# mid-stream thinking split live on the mate-e2b-kaggle branch; clone the
# branch explicitly so the kernel always runs the intended code.
res = subprocess.run(["git", "clone", "--quiet", "--branch", "mate-e2b-kaggle",
                      url, str(REPO)],
                     capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError("clone failed (token not attached?): " + res.stderr[-300:])
os.chdir(REPO)
print("cwd:", Path.cwd())

## 3. Run the aggregator (forever)

In [ ]:
import subprocess, sys
cmd = [sys.executable, "scripts/aggregate_live_state.py",
       "--namespace", "gemma",
       "--run-id", "gemma-1000-campaign",
       "--workers", "w1,w2",
       "--interval", "45"]
print("running:", " ".join(cmd))
res = subprocess.run(cmd, stderr=subprocess.STDOUT)
if res.returncode != 0:
    raise RuntimeError(f"aggregator exited rc={res.returncode} -- see output above")

## Notes
- If a worker stops publishing, the aggregator keeps going with the remaining worker and shows a stale worker in its report.
- The dashboard: chess-bench-live.pages.dev/gemma.html